# 03 — Pretraining ablations (A3–A6) and the retrieval check

This notebook does not need anything from notebook 01. It pretrains the variants listed in `VARIANTS` (cell 4) **at the same time, one per GPU**. Choose the **GPU L4 ×4** accelerator (4 GPUs), or T4 ×2 if L4 is unavailable; extra variants wait for a free GPU.

| Variant | What is removed |
|---|---|
| `full` | nothing (reference) — **already finished** |
| `no_contrastive` | contrastive (InfoNCE) terms |
| `recon_only` | everything except reconstruction terms |
| `no_morph` | morphology terms |
| `raw_ecg` | wavelet ECG front-end (plain patches instead) |

To save time the ablations use a **shortened budget of 30 epochs** (~7 h on an L4), the same for every variant, including a new 30-epoch `full` as the fair reference. With L4 ×4 the three variants finish in **one commit**; with T4 ×2 expect two commits. The script stops cleanly at `TIME_LIMIT_H` (11 h) so the commit still saves its output. If the log ends with *"time limit: stopping"*, attach this notebook's own output (see inputs) and commit again with the same settings. Each variant continues where it stopped.

**Session options:** Accelerator **GPU**. Use the same GPU type for every commit (L4 ×4 preferred). Internet is not needed.

**Inputs to attach (+ Add Input):**
1. **`jbhi-experiments`**: the scripts (shared with you by the dataset owner, under *Datasets → Shared with you*).
2. **Your MIMIC-BP dataset**: the folder with `ecg/`, `ppg/`, `abp/` sub-folders of `.npy` files (`dataverse_files`).
3. **`pretrain-v6-weights`** (dataset owned by `phantom8`, shared with you; *Datasets → Shared with you*). It holds the thesis model's `hybrid_v4_encoders.pth` and `hybrid_v4_full.pth`. Use this one, not the `jbhi-pretrained-*` models: those are a different pretraining run.
4. **From commit 2 on: this notebook's own output.** *+ Add Input → Notebooks → Your Work → this notebook*. That brings back finished runs and any half-finished one. Keep it attached; adding the newest version each time is enough.

**How to run:** check `VARIANTS` in cell 4, then **Save Version → Save & Run All (Commit)**. After each commit, attach its output (input 4) and commit again with the same settings until cell 7 shows `finished = True` for every variant. Do not run training interactively: an idle interactive session is stopped after ~20 min.

## 1 — Find inputs, copy scripts, show the GPU

In [ ]:
import glob, os, shutil, json

def find(pattern):
    return sorted(glob.glob(pattern, recursive=True))

s = find('/kaggle/input/**/pretrain.py')
assert s, 'Scripts not found: attach the jbhi-experiments dataset'
shutil.copytree(os.path.dirname(s[0]), '/kaggle/working/exp', dirs_exist_ok=True)
os.chdir('/kaggle/working/exp')
print('scripts:', os.getcwd())

# MIMIC-BP: a folder with ecg/ ppg/ abp/ sub-folders that contain .npy files
MIMIC = None
for d in find('/kaggle/input/**/ecg/'):
    root = os.path.dirname(os.path.normpath(d))
    if all(os.path.isdir(os.path.join(root, m)) for m in ['ppg', 'abp']) and \
            any(f.endswith('.npy') for f in os.listdir(d)[:200]):
        MIMIC = root
        break
assert MIMIC, 'MIMIC-BP not found: attach the dataset with ecg/ ppg/ abp/ .npy folders'
print('MIMIC-BP:', MIMIC, {m: len(os.listdir(os.path.join(MIMIC, m))) for m in ['ecg', 'ppg', 'abp']})

# the thesis model = the copy in phantom8's pretrain-v6-weights (verified against the thesis fold models)
def pick(name):
    hits = find(f'/kaggle/input/**/{name}')
    thesis = [h for h in hits if 'pretrain-v6-weights' in h]
    return (thesis or hits or [None])[0], bool(thesis)
ENC, enc_ok = pick('hybrid_v4_encoders.pth')
FULL, full_ok = pick('hybrid_v4_full.pth')
print('thesis encoders:', ENC, '' if enc_ok else '  <-- NOT the thesis copy (attach pretrain-v6-weights)')
print('thesis full    :', FULL, '' if full_ok else '  <-- NOT the thesis copy (attach pretrain-v6-weights)')
if not full_ok:
    FULL = None   # never run the thesis retrieval check on a different model

SAVE = '/kaggle/working/pretrain'
os.makedirs(SAVE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 — Architecture of the thesis model (so the ablations match it)

In [ ]:
PATCH_LEN, ECG = 25, 'wavelet'          # defaults if the weights are not attached
if ENC:
    from common import read_state_dict, describe_checkpoint
    info = describe_checkpoint(read_state_dict(ENC))
    for k, v in info.items():
        print(f'  {k:14s}: {v}')
    PATCH_LEN = info['enc_ppg']['patch_len']
    ECG = 'wavelet' if info['enc_ecg']['type'] == 'wavelet' else 'raw'
else:
    print('[!] hybrid_v4_encoders.pth not attached -- using defaults')
print(f'\n==> pretraining with patch_len={PATCH_LEN}, ECG branch={ECG}')

## 3 — Bring back earlier commits (finished or half-finished runs)

In [ ]:
prev = [p for p in find('/kaggle/input/**/pretrain_*') + find('/kaggle/input/**/retrieval_*')
        if os.path.isfile(p) and not p.startswith(SAVE)]
for p in prev:
    dst = os.path.join(SAVE, os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy2(p, dst)
print(f'restored {len(prev)} files from earlier commits')
for f in sorted(os.listdir(SAVE)):
    print('  ', f)

## 4 — Settings for THIS commit

In [ ]:
VARIANTS = ['full', 'no_contrastive', 'raw_ecg']   # run together, one per GPU
EPOCHS = 30                 # shortened budget, SAME for every variant (incl. a new 'full') -> fair comparison
DO_OLD_RETRIEVAL = False    # True only if the thesis-model retrieval result is missing
TIME_LIMIT_H = 11           # Kaggle stops commits at 12 h

FLAGS = {
    'full':           f'--variant full --ecg_encoder {ECG}',
    'no_contrastive': f'--variant no_contrastive --ecg_encoder {ECG}',
    'recon_only':     f'--variant recon_only --ecg_encoder {ECG}',
    'no_morph':       f'--variant no_morph --ecg_encoder {ECG}',
    'raw_ecg':        '--variant full --ecg_encoder raw',
}
TAG = {v: f'{v}_e{EPOCHS}' for v in VARIANTS}      # e.g. full_e30 (the 100-epoch 'full' stays untouched)
def finished(v):
    return os.path.exists(f'{SAVE}/pretrain_{TAG[v]}_retrieval.json')
for v in VARIANTS:
    print(f'{TAG[v]:20s} {"finished" if finished(v) else "to run"}  ->  {FLAGS[v]} --epochs {EPOCHS} --patch_len {PATCH_LEN}')

## 5 — Retrieval check on the thesis model (first commit only, ~15 min)

Ranks every validation window against **all** windows of the other modality, in two modes. *Joint* is how the thesis number was most likely computed: the shared encoder sees all three signals of the window, so it is optimistic. *Isolated* encodes each signal on its own; report this one. `--legacy_order` rebuilds the validation set in the original notebook's file order, so that it is (very likely) the same 4,572 windows the thesis model never trained on.

In [ ]:
if DO_OLD_RETRIEVAL and FULL:
    cmd = (f'python pretrain.py --mimic_dir "{MIMIC}" --compile_to /tmp/mimic_legacy --legacy_order '
           f'--retrieval_only "{FULL}" --patch_len {PATCH_LEN} --ecg_encoder {ECG} --save_dir {SAVE}')
    print(cmd)
    !{cmd}
    shutil.rmtree('/tmp/mimic_legacy', ignore_errors=True)
else:
    print('skipped')

## 6 — Pretraining runs for this commit (one variant per GPU)

In [ ]:
import subprocess, threading, queue, time, torch
T0 = time.time()
# compile MIMIC-BP once (same sorted order as the finished 'full' run), then share it
from pretrain import compile_mimic_to_memmap
compile_mimic_to_memmap(MIMIC, '/tmp/mimic_memmap')

N_GPU = max(1, torch.cuda.device_count())
LOGS = f'{SAVE}/_logs'
os.makedirs(LOGS, exist_ok=True)
todo = [v for v in VARIANTS if not finished(v)]
print(f'{N_GPU} GPU(s); running: {todo}')
jobs = queue.Queue()
for v in todo:
    jobs.put(v)

def worker(gpu):
    while True:
        try:
            v = jobs.get_nowait()
        except queue.Empty:
            return
        hours_left = TIME_LIMIT_H - (time.time() - T0) / 3600
        if hours_left < 0.5:
            print(f'[gpu{gpu}] no time left to start {v}', flush=True)
            continue
        cmd = (f'python pretrain.py --compiled_dirs /tmp/mimic_memmap {FLAGS[v]} --epochs {EPOCHS} '
               f'--patch_len {PATCH_LEN} --tag {TAG[v]} --save_dir {SAVE} --resume --time_limit_h {hours_left:.2f}')
        print(f'[gpu{gpu}] start {v}: {cmd}', flush=True)
        with open(f'{LOGS}/{TAG[v]}.log', 'a') as lf:
            rc = subprocess.run(cmd, shell=True, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)),
                                stdout=lf, stderr=subprocess.STDOUT).returncode
        print(f'[gpu{gpu}] end {v} (exit code {rc})', flush=True)
        if rc != 0:
            print(''.join(open(f'{LOGS}/{TAG[v]}.log').readlines()[-30:]), flush=True)

threads = [threading.Thread(target=worker, args=(g,)) for g in range(N_GPU)]
for t in threads:
    t.start()
while any(t.is_alive() for t in threads):
    for t in threads:
        t.join(timeout=1800)                 # status every 30 min
    print(f'\n--- {(time.time() - T0) / 3600:.1f} h ---', flush=True)
    for v in todo:
        f = f'{LOGS}/{TAG[v]}.log'
        ep = [l.strip() for l in open(f) if l.strip().startswith('ep ')] if os.path.exists(f) else []
        print(f'  {TAG[v]:20s} {ep[-1] if ep else "starting"}', flush=True)
print('all variants for this commit have stopped')

## 7 — Summary of everything finished so far

In [ ]:
import pandas as pd
rows = []
for h in sorted(glob.glob(f'{SAVE}/pretrain_*_history.json')):
    tag = os.path.basename(h)[len('pretrain_'):-len('_history.json')]
    hist = json.load(open(h))
    best = min(hist, key=lambda r: r['val'])
    rows.append(dict(variant=tag, epochs_run=len(hist), best_epoch=best['epoch'], best_val=round(best['val'], 4),
                     ppg_mse=round(best.get('ppg_mse', float('nan')), 4), abp_mse=round(best.get('abp_mse', float('nan')), 4),
                     ecg_mse=round(best.get('ecg_mse', float('nan')), 4),
                     finished=os.path.exists(f'{SAVE}/pretrain_{tag}_retrieval.json')))
print(pd.DataFrame(rows).to_string(index=False) if rows else 'no runs yet')

for r in sorted(glob.glob(f'{SAVE}/*retrieval*.json')):
    res = json.load(open(r))
    print(f'\n{os.path.basename(r)}')
    print(pd.DataFrame(res).T[['R1', 'R5', 'MRR', 'pool']].round(3).to_string())

## 8 — Save

**Save Version → Save & Run All (Commit)**. The output folder `pretrain/` holds, per variant: `pretrain_<variant>_encoders.pth` (used for fine-tuning later), `_full.pth`, `_state.pth` (for resuming), `_history.json` (loss curves) and `_retrieval.json`.

**Next commit:** attach this notebook's newest output (input 4) and commit again with the same `VARIANTS`. Each variant continues where it stopped. When cell 7 shows `finished = True` for all three variants, share the notebook with the project owner.

In [ ]:
os.chdir('/kaggle/working')
shutil.rmtree('/kaggle/working/exp', ignore_errors=True)
!ls -la {SAVE}